In [1]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, avg, count, when, sum
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from snowflake.snowpark import Session
from credentials import params

from snowflake.ml.modeling.impute import SimpleImputer
from snowflake.ml.modeling.preprocessing import OneHotEncoder
from snowflake.ml.modeling.pipeline import Pipeline

from snowflake.snowpark.types import (
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType
)

from snowflake.ml.registry import Registry

In [2]:
session = Session.builder.configs(params).create()
session.use_database("HOUSING_PRICE_PROJECT")
session.use_schema("ML_LAYER")

In [3]:
registry = Registry(
    session=session,
    database_name="HOUSING_PRICE_PROJECT",
    schema_name="ML_LAYER"
)

In [5]:
X_train = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET")
X_train = X_train.drop('PRICE_IN_LAKHS')

# robe per iperparam tuning

In [ ]:
res

# ***** FINE ROBE PER IPERPARAM TUNING *****

In [ ]:
#X_train.columns

In [ ]:
"""
X_train_long = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET")
#X_train = X_train.drop('"price_in_lakhs"', '"property_id"', '"price_category"', 'SOURCE_FILE')

for col in X_train_long.columns:
    if col[0] =='"':
        X_train_long = X_train_long.with_column_renamed(col, col[1:-1

X_train_long.columns
"""

In [ ]:
#X_train.write.mode("overwrite").save_as_table(
#    "HOUSING_PRICE_PROJECT.ML_LAYER.X_TRAIN_REDUCED"
#)

In [ ]:
#X_train = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.X_TRAIN_REDUCED")

In [ ]:
#X_train = X_train_2
#X_train.columns

# categorical feature cardinality

In [ ]:
#cat_col = list(X_train.select_dtypes(exclude=['number']).columns)
#cat_col.remove('property_id')
#cat_col.remove('price_category')
#cat_col.remove('SOURCE_FILE')
#cat_col

In [ ]:
#for i in cat_col:
#    print(f"{i}  ->  {df[i].nunique()}")

In [ ]:
#X_train[cat_col[1]].value_counts()

ottimo, prossimi steps:
1. pipeline con gli steps 2 e 3
2. fare imputation dei missing values
3. one hot-encoding di tutte le categorical features
4. fare training di XGBEstimator, distributed ML training

# Preprocessing

In [ ]:
floating_types = (
    FloatType,
    DoubleType,
    DecimalType
)

integer_types = (
    IntegerType,
    LongType
)
    
#************************************************************************
integer_cols = [
    field.name
    for field in X_train.schema.fields
    if isinstance(field.datatype, integer_types)    
]

#************************************************************************
floating_cols = [
    field.name
    for field in X_train.schema.fields
    if isinstance(field.datatype, floating_types)
]

#floating_cols.remove('"price_in_lakhs"')

#************************************************************************
numeric_cols = integer_cols + floating_cols

#************************************************************************
categorical_cols = [
    field.name
    for field in X_train.schema.fields
    if field.name not in numeric_cols    
]
#categorical_cols.remove('"price_in_lakhs"')
#categorical_cols.remove('"property_id"')
#categorical_cols.remove('"price_category"')
#categorical_cols.remove('SOURCE_FILE')

#************************************************************************
#categorical_cols_output = categorical_cols
#categorical_cols_output = []
#for cat in categorical_cols:
#    categorical_cols_output.append(cat[1:-1])

In [ ]:
categorical_cols

In [ ]:
imputer_num_int = SimpleImputer(
    input_cols = integer_cols,
    output_cols = integer_cols,
    strategy = 'median'
)

imputer_num_float = SimpleImputer(
    input_cols = floating_cols,
    output_cols = floating_cols,
    strategy = 'median'
)

imputer_cat = SimpleImputer(
    input_cols = categorical_cols,
    output_cols = categorical_cols,
    strategy = 'most_frequent'
)

encoder = OneHotEncoder(
    input_cols = categorical_cols,
    output_cols = categorical_cols,
    drop_input_cols = True,
    handle_unknown = 'ignore'
)

prepr_pip = Pipeline(
    steps = [
        ('imputer_num', imputer_num_int),
        ('imputer_num_float', imputer_num_float),
        ('imputer_cat', imputer_cat),
        ('encoder', encoder)
    ]
)

In [ ]:
prepr_pip.fit(X_train)

In [ ]:
#prepr_xtrain_sp = prepr_pip.transform(X_train)
#prepr_xtrain_sp.columns

In [ ]:
registry.log_model(
    model=prepr_pip,
    model_name="PREPROCESSING_PIPELINE",
    sample_input_data=X_train.limit(10)
    #, comment = "pipeline senza one-hot encoding"
)

In [ ]:
model_version = registry \
    .get_model("PREPROCESSING_PIPELINE") \
    .version("LAST")

predictions = model_version.run(
    X_train, #X_train_long
    function_name="transform"
)

In [ ]:
predictions.show()

In [ ]:
predictions.columns

In [ ]:
model = registry.get_model("PREPROCESSING_PIPELINE")
model.show_versions()

In [ ]:
model = registry.get_model("PREPROCESSING_PIPELINE")
model.default = model.version("LAST")

In [ ]:
#predictions = model_version.run(
#    X_train_long,
#    function_name="transform"
#)

#predictions.show()

In [ ]:
#predictions.columns

# default model

In [4]:
model = registry.get_model("PREPROCESSING_PIPELINE")
model.show_versions()

,created_on,name,aliases,comment,database_name,schema_name,model_name,is_default_version,functions,metadata,user_data,model_attributes,size,environment,runnable_in,inference_services
0,2026-08-31 09:23:53.104000-07:00,RUN_20260831_092255,"[""DEFAULT"",""FIRST"",""LAST""]",None,HOUSING_PRICE_PROJECT,ML_LAYER,PREPROCESSING_PIPELINE,true,"[""TRANSFORM""]",{},{},"{""framework"":""snowml"",""client"":""snowflake-ml-p...",43437,"{""default"":{""python_version"":""3.12"",""snowflake...","[""WAREHOUSE"",""SNOWPARK_CONTAINER_SERVICES""]",[]


In [6]:
model_version = registry \
    .get_model("PREPROCESSING_PIPELINE") \
    .version("DEFAULT")

predictions = model_version.run(
    X_train,
    function_name="transform"
)

In [7]:
predictions.columns

['CITY',
 'LOCALITY',
 'LOCALITY_TIER',
 'PROPERTY_TYPE',
 'FLOOR_CATEGORY',
 'FACING',
 'FURNISHING_STATUS',
 'TRANSACTION_TYPE',
 'BALCONIES',
 'CARPET_AREA',
 'FLOOR_NUMBER',
 'TOTAL_FLOORS',
 'PROPERTY_AGE',
 'PARKING_SPACES',
 'SECURITY_SCORE',
 'GYM_AVAILABLE',
 'SWIMMING_POOL',
 'POWER_BACKUP',
 'LIFT_AVAILABLE',
 'MAINTENANCE_FEE_MONTHLY',
 'DISTANCE_TO_CITY_CENTER_KM',
 'DISTANCE_TO_METRO_KM',
 'NEARBY_SCHOOLS',
 'NEARBY_HOSPITALS',
 '"""CITY_Bangalore"""',
 '"""CITY_Delhi"""',
 '"""CITY_Hyderabad"""',
 '"""CITY_Mumbai"""',
 '"""LOCALITY_Andheri"""',
 '"""LOCALITY_Bandra"""',
 '"""LOCALITY_Banjara Hills"""',
 '"""LOCALITY_Dwarka"""',
 '"""LOCALITY_Electronic City"""',
 '"""LOCALITY_Gachibowli"""',
 '"""LOCALITY_Gurgaon DLF"""',
 '"""LOCALITY_HSR Layout"""',
 '"""LOCALITY_Indiranagar"""',
 '"""LOCALITY_Jubilee Hills"""',
 '"""LOCALITY_Kondapur"""',
 '"""LOCALITY_Koramangala"""',
 '"""LOCALITY_Madhapur"""',
 '"""LOCALITY_Navi Mumbai"""',
 '"""LOCALITY_Noida Extension"""',
 '"""L

In [ ]:
#result = model_version.run(
#    X_train.limit(1),
#    function_name="transform"
#)

#result.explain()

# Altro

In [ ]:
#encoder = OneHotEncoder(
#    input_cols = categorical_cols,
#    output_cols = categorical_cols_output,
#    drop_input_cols = True,
#    handle_unknown = 'ignore'
#)

#encoder.fit(X_train)

#a = encoder.transform(X_train)

In [ ]:
#a.columns                     

In [ ]:
#a.show()

In [ ]:
#registry.log_model(
#    model=encoder,
#    model_name="PREPROCESSING_PIPELINE_OTHER",
#    sample_input_data=X_train.limit(10)
#)

In [ ]:
#model_version = registry \
#    .get_model("PREPROCESSING_PIPELINE_OTHER") \
#    .version("STALE_SHRIMP_3")

#predictions = model_version.run(
#    X_train,
#    function_name="transform"
#)

In [ ]:
#predictions.show()

In [ ]:
#data = [
#    ("Delhi", "Apartment"),
#    ("Mumbai", "Villa"),
#    ("Delhi", "Villa"),
#    ("Bangalore", "Apartment"),
#    ("Delhi", "Apartment"),
#]

#df_test = session.create_dataframe(
#    data,
#    schema=["CITY", "PROPERTY_TYPE"]
#)

#df_test.show()

In [ ]:
#encoder = OneHotEncoder(
#    input_cols = ["CITY", "PROPERTY_TYPE"],
#    output_cols = ['c', 'pt'],
#    #drop_input_cols = True,
#    handle_unknown = 'ignore'
#)

#encoder.fit(df_test)
#a = encoder.transform(df_test)
#a.show()

In [ ]:
#df_test.columns

In [ ]:
#a.columns